# Treino guiado de especialista — Kaggle

Fluxo: dataset causal → verificação de causalidade → regra-professora (escolhida só no treino) → clonagem de comportamento + ajuste fino por RL → seleção pela validação → holdout uma única vez.

**Nunca** envie `.env`, chaves de API ou tokens para este notebook.


In [ ]:
# ---------------------------------------------------------------- CONFIG
AGENT = "bear"          # bull | bear | ranger
FINETUNE_STEPS = 120_000
# -------------------------------------------------------------------------
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    # A politica MLP treina bem em CPU; GPU so acelera.
    print('sem GPU: o treino segue em CPU')


In [ ]:
# O cd para a pasta pai vem ANTES do rm: numa reexecucao o kernel ja esta
# dentro do clone, e apagar o diretorio atual faz o git clone falhar.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --depth 1 https://github.com/drtassio/BinanceFuturesTrader.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r cloud/requirements-cloud.txt
import os
assert os.path.exists('cloud/train_guided.py'), 'clone falhou: pare aqui'


In [ ]:
import subprocess, sys
for script in ('scripts/build_causal_dataset.py', 'scripts/build_meta_features.py', 'scripts/verify_causality.py'):
    # A verificacao de causalidade falha se alguma feature enxergar o futuro.
    assert subprocess.run([sys.executable, script]).returncode == 0, f'{script} falhou'


In [ ]:
# A regra-professora e escolhida SO no bloco de treino e versionada no
# repositorio. Como o dataset e reconstruido de forma deterministica, ela
# vale igual aqui; so e recalculada se o arquivo nao existir.
import os, subprocess, sys
rule = f'models_ai/{AGENT}_edge_rule.json'
if not os.path.exists(rule):
    assert subprocess.run([sys.executable, 'scripts/tune_edge_rule.py', '--agent', AGENT,
                           '--workers', str(os.cpu_count() or 2)]).returncode == 0, 'nenhuma regra lucrativa'
print(open(rule).read())


In [ ]:
import subprocess, sys
cmd = [sys.executable, 'cloud/train_guided.py', '--agent', AGENT, '--finetune-steps', str(FINETUNE_STEPS)]
print(' '.join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    # Mostra so o progresso relevante; o log completo fica em logs/.
    if not (' - INFO - ' in line or ' - DEBUG - ' in line):
        print(line, end='')
print('exit code:', process.wait())
assert process.returncode == 0, 'treino falhou'


In [ ]:
import json, pathlib
report = json.loads(pathlib.Path(f'cloud/artifacts/{AGENT}_guided_report.json').read_text())
v = report['verdict']
print('APROVADO PARA OPERAR:', v['approved_for_live_trading'])
for name, passed in v['checks'].items():
    print('  [%s] %s' % ('OK ' if passed else 'NAO', name))
def line(m):
    return 'trades=%d retorno=%+.2f%% PF=%.2f maxDD=%.1f%%' % (
        int(m.get('num_trades', 0)), m.get('total_return_pct', 0) * 100,
        m.get('profit_factor', 0), m.get('max_drawdown_pct', 0) * 100)
print()
print('selecionada :', report['selected'])
print('validacao   :', line(report['selected_validation']))
print('holdout     :', line(report['holdout_metrics']))
print('buy & hold  : %+.2f%%' % (v['buy_and_hold_return'] * 100))


In [ ]:
# Empacota a pasta da execucao inteira: politica, scaler e contrato de
# features precisam viajar juntos. No seu PC:
#   python scripts/promote_model.py --agent <agente> --archive <arquivo baixado>
import pathlib, tarfile, datetime
runs = sorted(p for p in pathlib.Path('cloud/artifacts').glob(f'{AGENT}_guided_*')
              if (p / 'models' / f'{AGENT}_specialist_sac.zip').exists())
assert runs, 'nenhuma execucao com modelo salvo'
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%S')
archive = f'/kaggle/working/{AGENT}_guided_{stamp}.tar.gz'
with tarfile.open(archive, 'w:gz') as tar:
    for item in runs[-1].iterdir():
        if item.name != 'checkpoints':
            tar.add(item, arcname=item.name)
print('pacote:', archive)
print('baixe o pacote pela aba Output do Kaggle')
